# Парсинг дополнительных источников по товарам «Смешарики»

В этом ноутбуке собираются товары с дополнительных сайтов, которые не входят в основной датасет Wildberries и Ozon.

Собираются данные с трёх источников:

- Riki Collection / SASONKO;
- «Два мяча»;
- Smlerch.

Для сбора используется Selenium в Google Colab и BeautifulSoup для извлечения данных из HTML-кода страницы.

## 1. Установка библиотек

Устанавливаются библиотеки для автоматизации браузера и обработки HTML-страниц в Google Colab.

In [1]:
!pip install selenium google-colab-selenium pandas bs4 -q

print('Библиотеки установлены')

Библиотеки установлены


## 2. Импорт библиотек

Подключаются библиотеки для работы с браузером, HTML-кодом, таблицами и сохранением файлов.

In [2]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from google.colab import files
import google_colab_selenium as gs

ModuleNotFoundError: No module named 'google'

## 3. Парсинг Riki Collection / SASONKO

Открывается каталог сайта Riki Collection. Страница прокручивается вниз, чтобы подгрузились все карточки товаров. После этого HTML-код страницы обрабатывается через BeautifulSoup, из карточек извлекаются название, артикул, категория, цена, материалы, ссылка и фото.

In [ ]:
print('Запускаем браузер')

driver = gs.Chrome()

url = 'https://www.rikicollection.com/catalog'
driver.get(url)

print('Сайт открыт. Начинаем прокрутку страницы')

last_height = driver.execute_script('return document.body.scrollHeight')
scroll_attempts = 0

while True:
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(3)

    try:
        button = driver.find_element(By.CSS_SELECTOR, '.t-store__load-more-btn')

        if button.is_displayed():
            driver.execute_script('arguments[0].click();', button)
            time.sleep(2)
    except Exception:
        pass

    new_height = driver.execute_script('return document.body.scrollHeight')

    if new_height == last_height:
        scroll_attempts += 1

        if scroll_attempts >= 3:
            print('Конец страницы достигнут')
            break
    else:
        scroll_attempts = 0
        last_height = new_height

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, 'html.parser')
products = soup.find_all('div', class_='js-product')

print(f'Найдено товаров: {len(products)}')

In [ ]:
all_products = []

for product in products:
    name_tag = product.find(class_='js-product-name')
    name = name_tag.text.strip() if name_tag else 'Не указано'

    sku_tag = product.find(class_='js-product-sku')
    sku = sku_tag.text.strip() if sku_tag else 'Не указан'

    descr_tag = product.find(class_='js-store-prod-descr')
    descr = descr_tag.text.strip() if descr_tag else ''

    price_tag = product.find(class_='js-product-price')
    price = price_tag.text.strip().replace(' ', '') if price_tag else '0'

    product_url = product.get('data-product-url', '')

    if not product_url and product.find('a'):
        product_url = product.find('a').get('href', '')

    img_tag = product.find(class_='js-product-img')
    img_url = img_tag.get('data-original', '') if img_tag else ''

    options_tags = product.select('.js-product-edition-option-variants option')
    materials = [option.text.strip() for option in options_tags] if options_tags else []

    item_info = {
        'Название': name,
        'Артикул': sku,
        'Категория': descr,
        'Цена (руб)': price,
        'Материалы': ', '.join(materials),
        'Ссылка на товар': product_url,
        'Ссылка на фото': img_url
    }

    if item_info not in all_products:
        all_products.append(item_info)

print(f'Уникальных товаров собрано: {len(all_products)}')

df_riki = pd.DataFrame(all_products)
display(df_riki.head())

In [ ]:
filename = 'riki_selenium_fixed.xlsx'

df_riki.to_excel(filename, index=False)
files.download(filename)

print(f'Файл сохранён: {filename}')

## 4. Парсинг сайта «Два мяча»

С сайта «Два мяча» собираются кеды с персонажами «Смешариков». Персонаж определяется по ссылке на карточку товара, после чего для каждого товара сохраняются персонаж, название, цена, ссылка и изображение.

In [ ]:
driver = gs.Chrome()

url = 'https://dvamyacha.com/landing/Smeshariki'
driver.get(url)
time.sleep(5)

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, 'html.parser')
products = soup.find_all('div', class_='catalog-list-item')

print(f'Найдено карточек: {len(products)}')

In [ ]:
character_mapping = {
    '99': 'Крош',
    'z2': 'Пин',
    'g4': 'Лосяш',
    'x1': 'Нюша'
}

unique_products = {}

for item in products:
    name_tag = item.find('a', class_='catalog-list-item-name')

    if not name_tag:
        continue

    link = 'https://dvamyacha.com' + name_tag['href']

    if link in unique_products:
        continue

    price_tag = item.find('div', class_='catalog-list-item-price')

    if price_tag:
        price = price_tag.get_text(separator='|', strip=True).split('|')[0].strip()
    else:
        price = '0'

    img_tag = item.find('img', class_='catalog-list-item-img')
    img_url = 'https://dvamyacha.com' + img_tag['src'] if img_tag else ''

    character = 'Неизвестно'

    for key, name in character_mapping.items():
        if key in link:
            character = name
            break

    product_name = f'Кеды с {character}' if character != 'Неизвестно' else name_tag.text.strip()

    unique_products[link] = {
        'Персонаж': character,
        'Товар': product_name,
        'Цена': price,
        'Ссылка': link,
        'Фото': img_url
    }

df_dvamyacha = pd.DataFrame(list(unique_products.values()))

character_order = ['Крош', 'Пин', 'Лосяш', 'Нюша']
df_dvamyacha['Персонаж'] = pd.Categorical(
    df_dvamyacha['Персонаж'],
    categories=character_order,
    ordered=True
)

df_dvamyacha = df_dvamyacha.sort_values('Персонаж')

display(df_dvamyacha)

In [ ]:
filename = 'smeshariki_final.xlsx'

df_dvamyacha.to_excel(filename, index=False)
files.download(filename)

print(f'Файл сохранён: {filename}')

## 5. Парсинг Smlerch

На сайте Smlerch сначала открывается страница магазина и нажимается кнопка загрузки дополнительных товаров. Затем собираются ссылки на товары, в названии которых встречается слово «Смешарики». После этого код открывает каждую карточку и собирает подробные данные.

In [ ]:
driver = gs.Chrome()

url = 'https://smlerch.ru/#shop'
driver.get(url)
time.sleep(3)

while True:
    try:
        button = driver.find_element(By.CLASS_NAME, 'js-store-load-more-btn')

        if button.is_displayed():
            driver.execute_script('arguments[0].click();', button)
            time.sleep(3)
        else:
            break
    except Exception:
        break

soup = BeautifulSoup(driver.page_source, 'html.parser')

product_links = []

for item in soup.select('.js-product'):
    name_tag = item.select_one('.js-product-name')

    if name_tag and 'смешарики' in name_tag.text.lower():
        link = item.find('a')['href']
        product_links.append(link)

print(f'Найдено товаров: {len(product_links)}')

In [ ]:
all_data = []

for link in product_links:
    driver.get(link)
    time.sleep(2)

    product_soup = BeautifulSoup(driver.page_source, 'html.parser')

    name_tag = product_soup.select_one('.js-product-name')
    price_tag = product_soup.select_one('.js-store-prod-price-val')
    description_tag = product_soup.select_one('.t-store__prod-popup__text')

    name = name_tag.text.strip() if name_tag else ''
    price = price_tag.text.strip() if price_tag else '0'
    description = description_tag.text.strip() if description_tag else 'Нет описания'

    all_data.append({
        'Название': name,
        'Цена': price,
        'Описание': description,
        'Ссылка': link
    })

    print(f'Собран товар: {name}')

driver.quit()

df_smlerch = pd.DataFrame(all_data)
display(df_smlerch.head())

In [ ]:
filename = 'smlerch_full_data.xlsx'

df_smlerch.to_excel(filename, index=False)
files.download(filename)

print(f'Файл сохранён: {filename}')

## Итог

В результате работы ноутбука создаются три Excel-файла:

- `riki_selenium_fixed.xlsx` — товары Riki Collection / SASONKO;
- `smeshariki_final.xlsx` — товары сайта «Два мяча»;
- `smlerch_full_data.xlsx` — товары Smlerch.

Далее эти файлы можно объединить с основным датасетом маркетплейсов в отдельном ноутбуке объединения данных.